#### npz File MFCC

In [3]:
import os
import librosa
import numpy as np

ulazni_dir = "datasetSpeech_1"
fajlovi = [f for f in os.listdir(ulazni_dir) if f.endswith(".wav")]

# treci broj u imenu fajla
EMOTION_MAP = {
    "01": 0,  # neutral
    "02": 1,  # calm
    "03": 2,  # happy
    "04": 3,  # sad
    "05": 4,  # angry
    "06": 5,  # fearful
    "07": 6,  # disgust
    "08": 7,  # surprised
}

X = []
y = []

target_sr = 16000
n_mfcc = 40 

print(f"MFCC ekstrakcija {len(fajlovi)} fajlova sa {(n_mfcc)} koeficijenata")

for i, fajl in enumerate(fajlovi):
    try:

        deli = fajl.split("-")
        emocija_kod = deli[2]
        labela = EMOTION_MAP[emocija_kod]

        putanja = os.path.join(ulazni_dir, fajl)
        audio, sr = librosa.load(putanja, sr=target_sr, mono=True)

        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

        X.append(mfcc)
        y.append(labela)

    except Exception as e:
        print(f"greska na fajlu {fajl}: {e}")

    if (i + 1) % 200 == 0 or (i + 1) == len(fajlovi):
        print(f"obradjeno {i + 1}/{len(fajlovi)}")


X = np.array(X)
y = np.array(y)
X = np.expand_dims(X, axis=-1)

print("\nKRAJ")
print(f"Oblik ulaznih podataka (X): {X.shape}") 
print(f"Oblik labela (y):           {y.shape}")

np.savez_compressed("dataset_mfcc.npz", X=X, y=y)
print("Sačuvano u 'dataset_mfcc.npz'!")

MFCC ekstrakcija 1440 fajlova sa 40 koeficijenata
obradjeno 200/1440
obradjeno 400/1440
obradjeno 600/1440
obradjeno 800/1440
obradjeno 1000/1440
obradjeno 1200/1440
obradjeno 1400/1440
obradjeno 1440/1440

KRAJ
Oblik ulaznih podataka (X): (1440, 40, 110, 1)
Oblik labela (y):           (1440,)
Sačuvano u 'dataset_mfcc.npz'!


In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Setovanje uređaja i seed-a
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")


def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    """Primenjuje SpecAugment na MFCC spektrogram (40 koeficijenata)."""
    augmented = mfcc_spec.copy()

    # Provera i prilagođavanje dimenzija ako spektrogram ima kanal na kraju
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    # Frequency Masking (maskiranje MFCC koeficijenata)
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    # Time Masking (maskiranje vremenskih ramova)
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 1. Učitavanje MFCC skupa podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]

X_tr, y_tr = [], []
X_va, y_va = [], []
X_te, y_te = [], []

# 2. Podela po klasama i augmentacija neutralne klase (c == 0)
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        # Originalnih 64 uzoraka
        X_orig = X_all[train_idx]

        # Generisanje 64 augmentisana uzorka primenom SpecAugment-a
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        # Spajanje 64 originalna + 64 augmentisana = 128 uzoraka
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])

    else:  # Ostale klase (imaju po 128 trening uzoraka)
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])

# 3. Spajanje i nasumično mešanje (permutation)
X_train = np.concatenate(X_tr, axis=0)
y_train = np.concatenate(y_tr, axis=0)
train_perm = np.random.permutation(len(y_train))
X_train, y_train = X_train[train_perm], y_train[train_perm]

X_val = np.concatenate(X_va, axis=0)
y_val = np.concatenate(y_va, axis=0)
val_perm = np.random.permutation(len(y_val))
X_val, y_val = X_val[val_perm], y_val[val_perm]

X_test = np.concatenate(X_te, axis=0)
y_test = np.concatenate(y_te, axis=0)
test_perm = np.random.permutation(len(y_test))
X_test, y_test = X_test[test_perm], y_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)

print("--- RASPODELA UZORAKA (MFCC SPEC-AUGMENT NEUTRAL) ---")
print(
    f"Train skup: {len(X_train)} uzoraka (128 po klasi - neutral augmentovan SpecAugment-om)"
)
print(f"Val skup:   {len(X_val)} uzoraka (32 za ostale, 16 za neutral)")
print(f"Test skup:  {len(X_test)} uzoraka (32 za ostale, 16 za neutral)\n")


# 5. PyTorch Dataset i DataLoader
class AudioDataset(Dataset):
    def __init__(self, X, y):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))  # Transformiše u (N, 1, 40, Time)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test), batch_size=32, shuffle=False
)

print(
    f"Oblik ulaznog tenzora u batch-u: {next(iter(train_loader_mfcc))[0].shape}"
)

Koristi se uređaj: cpu
--- RASPODELA UZORAKA (MFCC SPEC-AUGMENT NEUTRAL) ---
Train skup: 1024 uzoraka (128 po klasi - neutral augmentovan SpecAugment-om)
Val skup:   240 uzoraka (32 za ostale, 16 za neutral)
Test skup:  240 uzoraka (32 za ostale, 16 za neutral)

Oblik ulaznog tenzora u batch-u: torch.Size([32, 1, 40, 110])


#### 2N

In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc2N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = "model_mfcc_2N.pth"


# 1. SpecAugment funkcija za MFCC (40 koeficijenata)
def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

# 3. Podela po klasama, SpecAugment za neutralnu i dodela težina
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])

    else:
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)


# 5. PyTorch Dataset
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


# 6. Unapređena CNN arhitektura (16 -> 32 -> 64 filtera)
class MFCC_2CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_2CNN_Small, self).__init__()
        n=2
        self.features = nn.Sequential(
            nn.Conv2d(1,n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 <64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = (
            ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        )
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# Inicijalizacija mreže
model = MFCC_2CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(
    device
)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    scheduler.step(epoch_val_loss)

    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )

Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.0880 | Val Loss: 2.0781 | Val Acc: 10.83% | Test Loss: 2.0761 | Test Acc: 10.83% [Model sačuvan -> model_mfcc_2N.pth]
Epoha 02/30 | Train Loss: 2.0819 | Val Loss: 2.0741 | Val Acc: 12.50% | Test Loss: 2.0712 | Test Acc: 13.33% [Model sačuvan -> model_mfcc_2N.pth]
Epoha 03/30 | Train Loss: 2.0806 | Val Loss: 2.0718 | Val Acc: 11.25% | Test Loss: 2.0686 | Test Acc: 12.08% [Model sačuvan -> model_mfcc_2N.pth]
Epoha 04/30 | Train Loss: 2.0755 | Val Loss: 2.0706 | Val Acc: 14.17% | Test Loss: 2.0673 | Test Acc: 12.08% [Model sačuvan -> model_mfcc_2N.pth]
Epoha 05/30 | Train Loss: 2.0740 | Val Loss: 2.0689 | Val Acc: 15.83% | Test Loss: 2.0656 | Test Acc: 14.58% [Model sačuvan -> model_mfcc_2N.pth]
Epoha 06/30 | Train Loss: 2.0718 | Val Loss: 2.0673 | Val Acc: 14.58% | Test Loss: 2.0638 | Test Acc: 20.00% [Model sačuvan -> model_mfcc_2N.pth]
Epoha 07/30 | Train Loss: 2.0664 | Val Loss: 2.0655 | Val Acc: 16.67% | Test Loss: 2.0618 | Test Acc:

In [3]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_2CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_2N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

=== Predikcija za 8. fajl u test skupu ===
Stvarna emocija (Ground Truth): angry
Predviđena emocija:             angry (14.30%)
Vreme pojedinačne inferencije:  12.286 ms

Verovatnoće po klasama:
  neutral   :  10.27%
  calm      :  11.01%
  happy     :  12.89%
  sad       :  12.61%
  angry     :  14.30%
  fearful   :  13.79%
  disgust   :  12.97%
  surprised :  12.15%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 240
Ukupno trajanje:           0.0815 s
Prosečno vreme po uzorku:  0.340 ms
Brzina obrade (throughput): 2942.99 FPS (uzoraka/s)


#### 4N

In [5]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc4N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = "model_mfcc_4N.pth"


# 1. SpecAugment funkcija za MFCC (40 koeficijenata)
def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

# 3. Podela po klasama, SpecAugment za neutralnu i dodela težina
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])

    else:
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)


# 5. PyTorch Dataset
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


# 6. Unapređena CNN arhitektura (16 -> 32 -> 64 filtera)
class MFCC_4CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_4CNN_Small, self).__init__()
        n=4
        self.features = nn.Sequential(
            nn.Conv2d(1,n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 <64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = (
            ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        )
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# Inicijalizacija mreže
model = MFCC_4CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(
    device
)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    scheduler.step(epoch_val_loss)

    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )

Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.1060 | Val Loss: 2.0851 | Val Acc: 11.67% | Test Loss: 2.0862 | Test Acc: 14.17% [Model sačuvan -> model_mfcc_4N.pth]
Epoha 02/30 | Train Loss: 2.0863 | Val Loss: 2.0762 | Val Acc: 16.25% | Test Loss: 2.0766 | Test Acc: 15.83% [Model sačuvan -> model_mfcc_4N.pth]
Epoha 03/30 | Train Loss: 2.0760 | Val Loss: 2.0705 | Val Acc: 17.08% | Test Loss: 2.0698 | Test Acc: 17.92% [Model sačuvan -> model_mfcc_4N.pth]
Epoha 04/30 | Train Loss: 2.0648 | Val Loss: 2.0627 | Val Acc: 18.33% | Test Loss: 2.0607 | Test Acc: 17.92% [Model sačuvan -> model_mfcc_4N.pth]
Epoha 05/30 | Train Loss: 2.0654 | Val Loss: 2.0582 | Val Acc: 24.58% | Test Loss: 2.0555 | Test Acc: 25.42% [Model sačuvan -> model_mfcc_4N.pth]
Epoha 06/30 | Train Loss: 2.0556 | Val Loss: 2.0530 | Val Acc: 22.92% | Test Loss: 2.0488 | Test Acc: 23.75% [Model sačuvan -> model_mfcc_4N.pth]
Epoha 07/30 | Train Loss: 2.0435 | Val Loss: 2.0449 | Val Acc: 23.75% | Test Loss: 2.0394 | Test Acc:

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_4CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_4N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

#### 8N

In [6]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc8N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = "model_mfcc_8N.pth"


# 1. SpecAugment funkcija za MFCC (40 koeficijenata)
def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

# 3. Podela po klasama, SpecAugment za neutralnu i dodela težina
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])

    else:
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)


# 5. PyTorch Dataset
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


# 6. Unapređena CNN arhitektura (16 -> 32 -> 64 filtera)
class MFCC_8CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_8CNN_Small, self).__init__()
        n=8
        self.features = nn.Sequential(
            nn.Conv2d(1,n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 <64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = (
            ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        )
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# Inicijalizacija mreže
model = MFCC_8CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(
    device
)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    scheduler.step(epoch_val_loss)

    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )

Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.0797 | Val Loss: 2.0694 | Val Acc: 13.33% | Test Loss: 2.0682 | Test Acc: 13.33% [Model sačuvan -> model_mfcc_8N.pth]
Epoha 02/30 | Train Loss: 2.0713 | Val Loss: 2.0584 | Val Acc: 13.75% | Test Loss: 2.0561 | Test Acc: 13.33% [Model sačuvan -> model_mfcc_8N.pth]
Epoha 03/30 | Train Loss: 2.0577 | Val Loss: 2.0509 | Val Acc: 17.50% | Test Loss: 2.0490 | Test Acc: 20.42% [Model sačuvan -> model_mfcc_8N.pth]
Epoha 04/30 | Train Loss: 2.0467 | Val Loss: 2.0406 | Val Acc: 25.00% | Test Loss: 2.0387 | Test Acc: 27.50% [Model sačuvan -> model_mfcc_8N.pth]
Epoha 05/30 | Train Loss: 2.0367 | Val Loss: 2.0351 | Val Acc: 28.75% | Test Loss: 2.0318 | Test Acc: 31.67% [Model sačuvan -> model_mfcc_8N.pth]
Epoha 06/30 | Train Loss: 2.0263 | Val Loss: 2.0235 | Val Acc: 26.67% | Test Loss: 2.0192 | Test Acc: 27.50% [Model sačuvan -> model_mfcc_8N.pth]
Epoha 07/30 | Train Loss: 2.0120 | Val Loss: 2.0091 | Val Acc: 27.08% | Test Loss: 2.0027 | Test Acc:

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_8CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_8N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

#### 16N

In [7]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc16N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = "model_mfcc_16N.pth"


# 1. SpecAugment funkcija za MFCC (40 koeficijenata)
def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

# 3. Podela po klasama, SpecAugment za neutralnu i dodela težina
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])

    else:
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)


# 5. PyTorch Dataset
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


# 6. Unapređena CNN arhitektura (16 -> 32 -> 64 filtera)
class MFCC_16CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_16CNN_Small, self).__init__()
        n=16
        self.features = nn.Sequential(
            nn.Conv2d(1,n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 <64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = (
            ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        )
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# Inicijalizacija mreže
model = MFCC_16CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(
    device
)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    scheduler.step(epoch_val_loss)

    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )

Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.0801 | Val Loss: 2.0588 | Val Acc: 15.00% | Test Loss: 2.0546 | Test Acc: 19.58% [Model sačuvan -> model_mfcc_16N.pth]
Epoha 02/30 | Train Loss: 2.0475 | Val Loss: 2.0401 | Val Acc: 20.00% | Test Loss: 2.0311 | Test Acc: 22.08% [Model sačuvan -> model_mfcc_16N.pth]
Epoha 03/30 | Train Loss: 2.0169 | Val Loss: 2.0219 | Val Acc: 22.92% | Test Loss: 2.0093 | Test Acc: 26.25% [Model sačuvan -> model_mfcc_16N.pth]
Epoha 04/30 | Train Loss: 1.9889 | Val Loss: 1.9834 | Val Acc: 32.50% | Test Loss: 1.9667 | Test Acc: 32.92% [Model sačuvan -> model_mfcc_16N.pth]
Epoha 05/30 | Train Loss: 1.9452 | Val Loss: 1.9462 | Val Acc: 27.92% | Test Loss: 1.9279 | Test Acc: 28.75% [Model sačuvan -> model_mfcc_16N.pth]
Epoha 06/30 | Train Loss: 1.8869 | Val Loss: 1.8974 | Val Acc: 25.83% | Test Loss: 1.8836 | Test Acc: 30.00% [Model sačuvan -> model_mfcc_16N.pth]
Epoha 07/30 | Train Loss: 1.8311 | Val Loss: 1.8170 | Val Acc: 32.92% | Test Loss: 1.7874 | Tes

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_16CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_16N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

#### 32N


In [8]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc32N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = "model_mfcc_32N.pth"


# 1. SpecAugment funkcija za MFCC (40 koeficijenata)
def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

# 3. Podela po klasama, SpecAugment za neutralnu i dodela težina
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])

    else:
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)


# 5. PyTorch Dataset
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


# 6. Unapređena CNN arhitektura (16 -> 32 -> 64 filtera)
class MFCC_32CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_32CNN_Small, self).__init__()
        n=32
        self.features = nn.Sequential(
            nn.Conv2d(1,n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 <64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = (
            ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        )
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# Inicijalizacija mreže
model = MFCC_32CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(
    device
)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    scheduler.step(epoch_val_loss)

    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )

Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.0563 | Val Loss: 2.0429 | Val Acc: 25.83% | Test Loss: 2.0389 | Test Acc: 24.58% [Model sačuvan -> model_mfcc_32N.pth]
Epoha 02/30 | Train Loss: 1.9947 | Val Loss: 1.9939 | Val Acc: 22.08% | Test Loss: 1.9893 | Test Acc: 20.42% [Model sačuvan -> model_mfcc_32N.pth]
Epoha 03/30 | Train Loss: 1.9120 | Val Loss: 1.8772 | Val Acc: 34.17% | Test Loss: 1.8747 | Test Acc: 31.25% [Model sačuvan -> model_mfcc_32N.pth]
Epoha 04/30 | Train Loss: 1.8012 | Val Loss: 1.9756 | Val Acc: 19.17% | Test Loss: 1.9982 | Test Acc: 18.33%
Epoha 05/30 | Train Loss: 1.7009 | Val Loss: 1.7434 | Val Acc: 36.25% | Test Loss: 1.7280 | Test Acc: 35.42% [Model sačuvan -> model_mfcc_32N.pth]
Epoha 06/30 | Train Loss: 1.6112 | Val Loss: 1.6499 | Val Acc: 38.33% | Test Loss: 1.6367 | Test Acc: 39.58% [Model sačuvan -> model_mfcc_32N.pth]
Epoha 07/30 | Train Loss: 1.5452 | Val Loss: 2.1511 | Val Acc: 20.00% | Test Loss: 2.2052 | Test Acc: 19.17%
Epoha 08/30 | Train Loss

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_32CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_32N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

#### 64N

In [10]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc64N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = "model_mfcc_64N.pth"


# 1. SpecAugment funkcija za MFCC (40 koeficijenata)
def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

# 3. Podela po klasama, SpecAugment za neutralnu i dodela težina
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])

    else:
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)


# 5. PyTorch Dataset
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


# 6. Unapređena CNN arhitektura (16 -> 32 -> 64 filtera)
class MFCC_64CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_64CNN_Small, self).__init__()
        n=64
        self.features = nn.Sequential(
            nn.Conv2d(1,n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 <64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = (
            ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        )
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# Inicijalizacija mreže
model = MFCC_64CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(
    device
)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    scheduler.step(epoch_val_loss)

    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )

Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.0251 | Val Loss: 2.0231 | Val Acc: 20.83% | Test Loss: 2.0078 | Test Acc: 17.50% [Model sačuvan -> model_mfcc_64N.pth]
Epoha 02/30 | Train Loss: 1.9055 | Val Loss: 1.9058 | Val Acc: 26.67% | Test Loss: 1.8949 | Test Acc: 25.00% [Model sačuvan -> model_mfcc_64N.pth]
Epoha 03/30 | Train Loss: 1.7448 | Val Loss: 1.8277 | Val Acc: 33.33% | Test Loss: 1.7906 | Test Acc: 31.67% [Model sačuvan -> model_mfcc_64N.pth]
Epoha 04/30 | Train Loss: 1.6148 | Val Loss: 1.7156 | Val Acc: 35.83% | Test Loss: 1.6710 | Test Acc: 38.33% [Model sačuvan -> model_mfcc_64N.pth]
Epoha 05/30 | Train Loss: 1.5339 | Val Loss: 1.9860 | Val Acc: 26.67% | Test Loss: 2.0259 | Test Acc: 23.33%
Epoha 06/30 | Train Loss: 1.3944 | Val Loss: 1.9328 | Val Acc: 31.67% | Test Loss: 1.8621 | Test Acc: 32.08%
Epoha 07/30 | Train Loss: 1.3428 | Val Loss: 1.7132 | Val Acc: 35.83% | Test Loss: 1.6632 | Test Acc: 37.50% [Model sačuvan -> model_mfcc_64N.pth]
Epoha 08/30 | Train Loss

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_64CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_64N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

#### 128N

In [11]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc128N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = "model_mfcc_128N.pth"


# 1. SpecAugment funkcija za MFCC (40 koeficijenata)
def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

# 3. Podela po klasama, SpecAugment za neutralnu i dodela težina
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])

    else:
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)


# 5. PyTorch Dataset
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


# 6. Unapređena CNN arhitektura (16 -> 32 -> 64 filtera)
class MFCC_128CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_128CNN_Small, self).__init__()
        n=128
        self.features = nn.Sequential(
            nn.Conv2d(1,n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 <64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = (
            ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        )
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# Inicijalizacija mreže
model = MFCC_128CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(
    device
)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    scheduler.step(epoch_val_loss)

    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )

Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.0077 | Val Loss: 2.0030 | Val Acc: 22.92% | Test Loss: 1.9924 | Test Acc: 25.83% [Model sačuvan -> model_mfcc_128N.pth]
Epoha 02/30 | Train Loss: 1.7876 | Val Loss: 2.3996 | Val Acc: 17.08% | Test Loss: 2.4345 | Test Acc: 17.92%
Epoha 03/30 | Train Loss: 1.6443 | Val Loss: 3.4274 | Val Acc: 14.17% | Test Loss: 3.5378 | Test Acc: 13.75%
Epoha 04/30 | Train Loss: 1.5375 | Val Loss: 4.1028 | Val Acc: 13.33% | Test Loss: 4.0302 | Test Acc: 13.75%
Epoha 05/30 | Train Loss: 1.4399 | Val Loss: 2.9835 | Val Acc: 15.83% | Test Loss: 2.8743 | Test Acc: 17.50%
Epoha 06/30 | Train Loss: 1.3014 | Val Loss: 3.7586 | Val Acc: 15.42% | Test Loss: 3.6968 | Test Acc: 16.67%
Epoha 07/30 | Train Loss: 1.2051 | Val Loss: 2.7665 | Val Acc: 20.42% | Test Loss: 2.7449 | Test Acc: 16.25%
Epoha 08/30 | Train Loss: 1.0952 | Val Loss: 1.4018 | Val Acc: 45.83% | Test Loss: 1.3722 | Test Acc: 47.08% [Model sačuvan -> model_mfcc_128N.pth]
Epoha 09/30 | Train Loss: 1

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_128CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_128N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

#### 256N

In [12]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc256N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = "model_mfcc_256N.pth"


# 1. SpecAugment funkcija za MFCC (40 koeficijenata)
def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

# 3. Podela po klasama, SpecAugment za neutralnu i dodela težina
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])

    else:
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)


# 5. PyTorch Dataset
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


# 6. Unapređena CNN arhitektura (16 -> 32 -> 64 filtera)
class MFCC_256CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_256CNN_Small, self).__init__()
        n=256
        self.features = nn.Sequential(
            nn.Conv2d(1,n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 <64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = (
            ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        )
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# Inicijalizacija mreže
model = MFCC_256CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(
    device
)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    scheduler.step(epoch_val_loss)

    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )

Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.0537 | Val Loss: 1.9584 | Val Acc: 23.75% | Test Loss: 1.9210 | Test Acc: 30.00% [Model sačuvan -> model_mfcc_256N.pth]
Epoha 02/30 | Train Loss: 1.7787 | Val Loss: 1.8358 | Val Acc: 29.58% | Test Loss: 1.8536 | Test Acc: 25.00% [Model sačuvan -> model_mfcc_256N.pth]
Epoha 03/30 | Train Loss: 1.5923 | Val Loss: 4.0928 | Val Acc: 13.33% | Test Loss: 4.0355 | Test Acc: 15.00%
Epoha 04/30 | Train Loss: 1.5301 | Val Loss: 1.9657 | Val Acc: 31.25% | Test Loss: 1.8787 | Test Acc: 33.33%
Epoha 05/30 | Train Loss: 1.4295 | Val Loss: 3.4386 | Val Acc: 17.50% | Test Loss: 3.3839 | Test Acc: 17.08%
Epoha 06/30 | Train Loss: 1.2991 | Val Loss: 2.8729 | Val Acc: 29.58% | Test Loss: 2.9042 | Test Acc: 28.33%
Epoha 07/30 | Train Loss: 1.1504 | Val Loss: 1.7277 | Val Acc: 37.08% | Test Loss: 1.7054 | Test Acc: 30.83% [Model sačuvan -> model_mfcc_256N.pth]
Epoha 08/30 | Train Loss: 1.0320 | Val Loss: 3.6539 | Val Acc: 22.08% | Test Loss: 3.4230 | Test 

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_256CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_256N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

#### 512N

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc512N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = "model_mfcc_512N.pth"


# 1. SpecAugment funkcija za MFCC (40 koeficijenata)
def spec_augment_mfcc(mfcc_spec, freq_mask_max=4, time_mask_max=12):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0 : f0 + f, :] = 0

    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje podataka
data = np.load("dataset_mfcc.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

# 3. Podela po klasama, SpecAugment za neutralnu i dodela težina
for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0:  # Neutralna klasa (balansiranje pomoću SpecAugment-a)
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment_mfcc(x) for x in X_orig])

        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])

    else:
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

# 4. Per-Coefficient Normalizacija (duž vremenske ose za svaki MFCC koeficijent)
mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)


# 5. PyTorch Dataset
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


# 6. Unapređena CNN arhitektura (16 -> 32 -> 64 filtera)
class MFCC_512CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_512CNN_Small, self).__init__()
        n=512
        self.features = nn.Sequential(
            nn.Conv2d(1,n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 <64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = (
            ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        )
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# Inicijalizacija mreže
model = MFCC_512CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(
    device
)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    scheduler.step(epoch_val_loss)

    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_512CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_512N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")